# **UTP 2 - MADALINE (MRI) untuk Klasifikasi Citra Digit**

**Waktu: 120 menit** | Sifat: open notebook, tanpa internet selain untuk mengunggah file | Dikerjakan individu

### Studi kasus
Sebuah sistem pemilah formulir harus membedakan dua **kelompok** digit tulisan tangan berukuran **8x8 piksel** (nilai keabuan 0-16):

| Kelompok | Digit | Target |
|---|---|---|
| A - digit berlengkung | 2, 3, 5, 8 | `+1` |
| B - digit bergaris lurus | 1, 4, 7, 9 | `-1` |

Berbeda dengan UTP sebelumnya (digit 0 vs 1 yang mudah dipisahkan), setiap kelompok berisi **beberapa digit yang bentuknya berbeda-beda**, sehingga satu batas linear tidak cukup. Inilah alasan MADALINE (beberapa ADALINE + unit OR) digunakan.

### File data
- `utp2_train.csv` - 1006 citra untuk pelatihan (akan dibagi lagi menjadi train dan validasi)
- `utp2_test.csv` - 432 citra test bersih
- `utp2_test_noisy.csv` - 432 citra test yang sama, tetapi 10% pikselnya dirusak (*salt-and-pepper*)

Kolom pertama `label` berisi digit asli (1-9), kolom berikutnya `1x1` sampai `8x8` berisi nilai piksel.

### Ketentuan
1. Gunakan `SEED = 42`; jangan mengubah sel bertanda **JANGAN DIUBAH**.
2. Lengkapi setiap bagian bertanda `# TODO`. Notebook harus dapat dijalankan dari awal (Run all) tanpa error.
3. Jawaban analisis **wajib** merujuk pada angka/gambar dari output notebook Anda sendiri.
4. Dilarang menggunakan library jaringan saraf. Implementasi manual dengan NumPy.

### Bobot penilaian
| Bagian | Bobot |
|---|---|
| Tugas 1 - Pembentukan target dan binarisasi | 10 |
| Tugas 2 - Fungsi aktivasi | 5 |
| Tugas 3 - Lapisan output OR | 5 |
| Tugas 4 - `train` dengan aturan MRI | 20 |
| Tugas 5 - `test` | 5 |
| Tugas 6 - Pemilihan hyperparameter | 5 |
| Soal analisis 1-5 (masing-masing 10) | 50 |

In [ ]:
#JANGAN DIUBAH
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
KELAS_A = [2, 3, 5, 8]   # target +1
KELAS_B = [1, 4, 7, 9]   # target -1

In [ ]:
#JANGAN DIUBAH - unggah utp2_train.csv, utp2_test.csv, utp2_test_noisy.csv
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass

In [ ]:
#JANGAN DIUBAH - memuat dan menampilkan contoh citra
train_df = pd.read_csv('utp2_train.csv')
test_df  = pd.read_csv('utp2_test.csv')
noisy_df = pd.read_csv('utp2_test_noisy.csv')
print('train:', train_df.shape, '| test:', test_df.shape, '| test noisy:', noisy_df.shape)
print('Distribusi digit (train):', train_df['label'].value_counts().sort_index().to_dict())

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for k, ax in enumerate(axes[0]):
    ax.imshow(test_df.iloc[k, 1:].to_numpy().reshape(8, 8), cmap='gray_r'); ax.set_title(f'{test_df.label[k]}'); ax.axis('off')
for k, ax in enumerate(axes[1]):
    ax.imshow(noisy_df.iloc[k, 1:].to_numpy().reshape(8, 8), cmap='gray_r'); ax.set_title(f'{noisy_df.label[k]} (noisy)', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## Tugas 1 - Target dan binarisasi (10 poin)

1. `make_target(labels)` - kembalikan array berisi `+1` untuk digit di `KELAS_A` dan `-1` untuk digit di `KELAS_B`. Gunakan `np.isin` (tanpa loop).
2. `binarize(X, t)` - ubah nilai piksel menjadi bipolar: `1` jika piksel **> t**, selain itu `-1`.

Kemudian bagi `utp2_train.csv` menjadi **train 80% / validasi 20%** secara stratified berdasarkan **digit asli** (`label`), dengan `random_state=SEED`.

In [ ]:
def make_target(labels):
    # TODO: lengkapi kode di sini
    return np.where(np.isin(labels, KELAS_A), 1, -1)

def binarize(X, t):
    # TODO: lengkapi kode di sini
    return np.where(X > t, 1, -1)

X_all  = train_df.iloc[:, 1:].to_numpy()
lab_all = train_df['label'].to_numpy()
X_test_raw,  lab_test  = test_df.iloc[:, 1:].to_numpy(),  test_df['label'].to_numpy()
X_noisy_raw, lab_noisy = noisy_df.iloc[:, 1:].to_numpy(), noisy_df['label'].to_numpy()

# Bagi X_all menjadi X_tr_raw, X_va_raw, lab_tr, lab_va (stratified berdasarkan lab_all)
# TODO: lengkapi kode di sini
X_tr_raw, X_va_raw, lab_tr, lab_va = train_test_split(X_all, lab_all, test_size=0.2, stratify=lab_all, random_state=SEED)

y_tr, y_va = make_target(lab_tr), make_target(lab_va)
y_test, y_noisy = make_target(lab_test), make_target(lab_noisy)
print('train:', X_tr_raw.shape, ' val:', X_va_raw.shape)
print('Proporsi +1 (train):', np.mean(y_tr == 1).round(3))
print('Contoh binarize t=4:', binarize(np.array([0, 4, 5, 16]), 4))   # harus [-1 -1  1  1]

## Tugas 2 - Fungsi aktivasi (5 poin)

Lengkapi `aktivasi(x)` secara **vektor** (bisa menerima skalar maupun array): `1` jika `x >= 0`, selain itu `-1`. Jangan memakai `np.vectorize`.

In [ ]:
def aktivasi(x):
    # TODO: lengkapi kode di sini
    return np.where(np.asarray(x) >= 0, 1, -1)

print(aktivasi(np.array([-2, -0.0001, 0, 3])))   # harus [-1 -1  1  1]

## Tugas 3 - Lapisan output OR (5 poin)

Pada MADALINE-MRI, bobot lapisan output **tidak dilatih**: semua bobot $v_j = 0{,}5$ dan bias $b_{out}$ dipilih agar unit output menghitung **OR** dari $n$ unit tersembunyi, yaitu $y = 1$ jika **minimal satu** $z_j = 1$, dan $y = -1$ jika semua $z_j = -1$.

Lengkapi `output_layer(n)` yang mengembalikan `(v, b_out)` agar sifat OR berlaku untuk **sembarang** $n \ge 1$. Turunkan rumus `b_out` sendiri (petunjuk: jika $k$ dari $n$ unit bernilai $+1$, maka $\sum v_j z_j = 0{,}5(2k - n)$). Uji otomatis di bawah harus mencetak `True` untuk semua $n$.

In [ ]:
def output_layer(n):
    # TODO: lengkapi kode di sini
    # sum(v_j z_j) = 0.5(2k - n). Syarat OR: k=0 -> 0.5(-n) + b_out < 0  => b_out < 0.5n
    #                                         k=1 -> 0.5(2-n) + b_out >= 0 => b_out >= 0.5n - 1
    # sehingga b_out dipilih di tengah rentang [0.5n - 1, 0.5n): b_out = 0.5n - 0.5
    v = np.full(n, 0.5)
    b_out = 0.5 * n - 0.5
    return v, b_out

#JANGAN DIUBAH - uji sifat OR untuk semua kombinasi z
import itertools
for n in [1, 2, 3, 4, 8]:
    v, b_out = output_layer(n)
    ok = all((aktivasi(np.dot(z, v) + b_out) == (1 if 1 in z else -1)) for z in itertools.product([-1, 1], repeat=n))
    print(f'n={n}: OR benar untuk semua kombinasi? {ok}')

## Tugas 4 - Pelatihan MADALINE dengan aturan MRI (20 poin)

Lengkapi `train(X, T, alpha, max_epoch, n, seed)` dengan ketentuan:

- Inisialisasi `w` berukuran `(jumlah_fitur, n)` dan bias hidden `b` berukuran `(n,)` dari uniform $[-0{,}1, 0{,}1]$ memakai `rng = np.random.default_rng(seed)`. Bobot/bias output diambil dari `output_layer(n)` dan **tidak diubah**.
- Setiap epoch, urutan sampel diacak dengan `rng.permutation`. Hitung jumlah sampel salah per epoch ke list `errors`. Hentikan pelatihan lebih awal jika dalam satu epoch **tidak ada** kesalahan.
- Untuk setiap sampel, hitung $z\_in_j = x \cdot w_{:,j} + b_j$, $z_j$ = aktivasi, lalu $y$. Jika $y \ne t$:
  - **t = +1**: perbarui **hanya satu** unit $J$ yang $z\_in_J$-nya **paling dekat ke 0**:
    $w_{:,J} \mathrel{+}= \alpha\,(1 - z\_in_J)\,x$, $\; b_J \mathrel{+}= \alpha\,(1 - z\_in_J)$
  - **t = -1**: perbarui **setiap** unit $k$ dengan $z\_in_k > 0$:
    $w_{:,k} \mathrel{+}= \alpha\,(-1 - z\_in_k)\,x$, $\; b_k \mathrel{+}= \alpha\,(-1 - z\_in_k)$

Kembalikan dictionary `{'w', 'b', 'v', 'b_out', 'errors'}`.

In [ ]:
def train(X, T, alpha=0.01, max_epoch=50, n=2, seed=SEED):
    rng = np.random.default_rng(seed)
    # TODO: lengkapi kode di sini
    w = rng.uniform(-0.1, 0.1, (X.shape[1], n))
    b = rng.uniform(-0.1, 0.1, n)
    v, b_out = output_layer(n)          # bobot/bias output tetap (tidak dilatih)
    errors = []
    for epoch in range(max_epoch):
        n_err = 0
        for i in rng.permutation(len(X)):
            x, t = X[i], T[i]
            z_in = x @ w + b
            z = aktivasi(z_in)
            y = aktivasi(z @ v + b_out)
            if y != t:
                n_err += 1
                if t == 1:
                    # perbarui hanya satu unit dengan z_in paling dekat ke 0
                    J = np.argmin(np.abs(z_in))
                    w[:, J] += alpha * (1 - z_in[J]) * x
                    b[J] += alpha * (1 - z_in[J])
                else:
                    # perbarui setiap unit dengan z_in > 0
                    for k in np.where(z_in > 0)[0]:
                        w[:, k] += alpha * (-1 - z_in[k]) * x
                        b[k] += alpha * (-1 - z_in[k])
        errors.append(n_err)
        if n_err == 0:
            break
    return {'w': w, 'b': b, 'v': v, 'b_out': b_out, 'errors': errors}

## Tugas 5 - Pengujian (5 poin)

Lengkapi `test(model, X)` secara **vektor (tanpa loop)** untuk seluruh baris `X`.

In [ ]:
def test(model, X):
    # TODO: lengkapi kode di sini
    z = aktivasi(X @ model['w'] + model['b'])
    return aktivasi(z @ model['v'] + model['b_out'])

#JANGAN DIUBAH
def calc_accuracy(a, b):
    return float(np.mean(np.ravel(a) == np.ravel(b)))

## Tugas 6 - Pemilihan hyperparameter (5 poin)

Untuk setiap kombinasi threshold binarisasi `t` dan jumlah hidden `n`, binarisasi data **train dan validasi** dengan `t` yang sama, latih model (`alpha=0.01, max_epoch=50`), lalu catat akurasi train, akurasi validasi, dan jumlah epoch. Simpan sebagai DataFrame `hasil`. Pilih `(t, n)` dengan **akurasi validasi tertinggi** (jika seri, ambil yang muncul lebih dulu). Simpan modelnya sebagai `best_model` dan pasangan parameternya sebagai `best_t`, `best_n`.

In [ ]:
records = []
best_model, best_t, best_n, best_val = None, None, None, -1

for t in [0, 4, 8, 12]:
    for n in [1, 2, 4, 8]:
        # TODO: lengkapi kode di sini
        Xtr_t, Xva_t = binarize(X_tr_raw, t), binarize(X_va_raw, t)
        m = train(Xtr_t, y_tr, alpha=0.01, max_epoch=50, n=n)
        acc_tr = calc_accuracy(test(m, Xtr_t), y_tr)
        acc_va = calc_accuracy(test(m, Xva_t), y_va)
        records.append({'t': t, 'n': n, 'acc_train': acc_tr, 'acc_val': acc_va, 'epoch': len(m['errors'])})
        if acc_va > best_val:
            best_val, best_model, best_t, best_n = acc_va, m, t, n

hasil = pd.DataFrame(records)
print(hasil.round(3).to_string(index=False))
print()
print(hasil.pivot(index='t', columns='n', values='acc_val').round(3))

In [ ]:
#JANGAN DIUBAH - evaluasi model terbaik
X_test, X_noisy = binarize(X_test_raw, best_t), binarize(X_noisy_raw, best_t)
pred_test, pred_noisy = test(best_model, X_test), test(best_model, X_noisy)
print(f'Model terbaik: t={best_t}, n={best_n}, epoch={len(best_model["errors"])}')
print('Akurasi train      :', round(calc_accuracy(test(best_model, binarize(X_tr_raw, best_t)), y_tr), 4))
print('Akurasi validasi   :', round(best_val, 4))
print('Akurasi test bersih:', round(calc_accuracy(pred_test, y_test), 4))
print('Akurasi test noisy :', round(calc_accuracy(pred_noisy, y_noisy), 4))

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(best_model['errors']) + 1), best_model['errors'], marker='o')
plt.xlabel('epoch'); plt.ylabel('jumlah salah (train)'); plt.title('Kesalahan per epoch - model terbaik')
plt.show()

## Eksperimen pendukung analisis

Jalankan sel-sel berikut **tanpa mengubah kode**.

In [ ]:
#JANGAN DIUBAH - Eksperimen A: bias output tetap 0.5 (seperti UTP sebelumnya) untuk berbagai n
for n in [1, 2, 3, 4, 8]:
    v = np.full(n, 0.5)
    hasil_k = []
    for k in range(n + 1):           # k = banyaknya unit hidden yang bernilai +1
        z = np.array([1] * k + [-1] * (n - k))
        hasil_k.append(int(aktivasi(z @ v + 0.5)))
    print(f'n={n}: output untuk k = 0..{n} unit aktif -> {hasil_k}')

In [ ]:
#JANGAN DIUBAH - Eksperimen B: menukar kelas positif (KELAS_B menjadi +1), t = best_t
Xtr_b, Xva_b, Xte_b = binarize(X_tr_raw, best_t), binarize(X_va_raw, best_t), binarize(X_test_raw, best_t)
rows = []
for n in [1, 2, 4, 8]:
    mA = train(Xtr_b,  y_tr, n=n)
    mB = train(Xtr_b, -y_tr, n=n)
    rows.append({'n': n,
                 'val (A = +1)': calc_accuracy(test(mA, Xva_b),  y_va),
                 'val (B = +1)': calc_accuracy(test(mB, Xva_b), -y_va),
                 'test (A = +1)': calc_accuracy(test(mA, Xte_b),  y_test),
                 'test (B = +1)': calc_accuracy(test(mB, Xte_b), -y_test)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
#JANGAN DIUBAH - Eksperimen C: aturan update pada UTP sebelumnya vs MRI (n = 4, t = best_t)
def train_lama(X, T, alpha=0.01, max_epoch=50, n=4, seed=SEED):
    rng = np.random.default_rng(seed)
    w = rng.random((X.shape[1], n))           # bobot awal positif semua
    v = np.full(n, 0.5)
    b = np.concatenate((np.full(n, 0.5), [0.5]))
    errors = []
    for _ in range(max_epoch):
        n_err = 0
        for data, target in zip(X, T):
            z_in = data @ w + b[:-1]
            z = aktivasi(z_in)
            y = aktivasi(z @ v + b[-1])
            if y != target:
                n_err += 1
                if target == 1:
                    w += alpha * np.outer(data, (1 - z))      # semua unit bernilai -1 ikut diubah
                else:
                    for i in np.where(z_in > 0)[0]:
                        w[:, i] -= alpha * data               # bias tidak pernah diubah
        errors.append(n_err)
        if n_err == 0:
            break
    return {'w': w, 'b': b[:-1], 'v': v, 'b_out': b[-1], 'errors': errors}

m_lama = train_lama(Xtr_b, y_tr)
m_mri  = train(Xtr_b, y_tr, n=4)
for nama, m in [('lama', m_lama), ('MRI ', m_mri)]:
    zt = aktivasi(Xte_b @ m['w'] + m['b'])
    print(f"{nama}: test={calc_accuracy(test(m, Xte_b), y_test):.3f}  error epoch 1/5/akhir={m['errors'][0]}/{m['errors'][min(4, len(m['errors'])-1)]}/{m['errors'][-1]}"
          f"  rata-rata unit aktif per citra test={np.mean(np.sum(zt == 1, axis=1)):.2f}")

In [ ]:
#JANGAN DIUBAH - Eksperimen D: visualisasi bobot unit hidden dan kesalahan per digit (model terbaik)
n_show = best_model['w'].shape[1]
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2.3), squeeze=False)
for j in range(n_show):
    ax = axes[0, j]
    lim = np.abs(best_model['w'][:, j]).max()
    ax.imshow(best_model['w'][:, j].reshape(8, 8), cmap='bwr', vmin=-lim, vmax=lim)
    aktif = np.mean(aktivasi(X_test @ best_model['w'][:, j] + best_model['b'][j]) == 1)
    ax.set_title(f'unit {j} (aktif {aktif:.0%})', fontsize=8); ax.axis('off')
plt.suptitle('Bobot hidden (merah = positif, biru = negatif)', fontsize=9); plt.tight_layout(); plt.show()

per_digit = pd.DataFrame({'digit': lab_test,
                          'salah_bersih': pred_test != y_test,
                          'salah_noisy': pred_noisy != y_noisy}).groupby('digit').mean().round(3)
print('Proporsi salah per digit (test):')
print(per_digit)

Z = aktivasi(X_test @ best_model['w'] + best_model['b'])
unit_per_digit = pd.DataFrame(Z == 1, columns=[f'unit {j}' for j in range(n_show)])
unit_per_digit['digit'] = lab_test
print()
print('Proporsi unit hidden aktif (+1) per digit (test bersih):')
print(unit_per_digit.groupby('digit').mean().round(2))

# **Soal Analisis** (masing-masing 10 poin)

Jawab setiap soal di sel jawaban di bawahnya dengan merujuk angka/gambar spesifik dari notebook Anda.

**Soal 1 - Konvergensi dan generalisasi.**
Laporkan `(t, n)` terbaik, akurasi train, validasi, test bersih, dan test noisy. (a) Berdasarkan grafik kesalahan per epoch, apakah pelatihan mencapai nol kesalahan? Jelaskan apa arti kondisi berhenti `weight_updated == False` pada UTP sebelumnya dan mengapa UTP sebelumnya berhenti hanya dalam 6 epoch. (b) UTP sebelumnya melaporkan akurasi 1,0. Sebutkan dua alasan mengapa angka itu tidak dapat dibandingkan langsung dengan akurasi test di UTP ini. (c) Berapa penurunan akurasi akibat derau 10%, dan apakah menurut Anda model cukup robust?

**Jawaban:**

Model terbaik adalah t = 8 dan n = 8 dengan akurasi train 1.0, validasi 0.9208, test bersih 0.9005, dan test noisy 0.8241. (a) Grafik kesalahan per epoch menunjukkan jumlah salah turun sampai nol pada epoch ke-36 (kurang dari batas 50 epoch), sehingga pelatihan mencapai nol kesalahan dan berhenti lebih awal; kondisi berhenti weight_updated == False pada UTP sebelumnya berarti dalam satu epoch penuh tidak ada bobot yang berubah, yaitu semua sampel sudah diklasifikasikan benar, dan UTP sebelumnya selesai hanya dalam 6 epoch karena datanya hanya 4 titik (XOR/AND) sehingga sangat mudah dihafal. (b) Akurasi 1,0 pada UTP sebelumnya tidak dapat dibandingkan langsung dengan akurasi test di sini karena pertama, akurasi itu dihitung pada data yang sama dengan data latih (hanya mengukur hafalan, sedangkan di sini test adalah 432 citra yang belum pernah dilihat), dan kedua, datanya hanya 4 titik tanpa variasi bentuk dan derau, sedangkan di sini tiap kelompok berisi beberapa digit dengan variasi tulisan tangan yang besar. (c) Derau 10% menurunkan akurasi dari 0.9005 menjadi 0.8241, yaitu turun sekitar 7.6 poin; menurut saya model cukup tetapi belum sangat robust, karena masih di atas 82% namun kesalahan pada digit kelompok B yang tadinya hampir tanpa salah (digit 7 dan 4 dari 0.000 menjadi 0.204 dan 0.123) meningkat tajam, sebab binarisasi dengan threshold membuat piksel rusak langsung membalik nilai fitur.

**Soal 2 - Jumlah unit hidden dan threshold binarisasi.**
Gunakan tabel/pivot hasil Tugas 6. (a) Bandingkan `n = 1` (setara satu ADALINE) dengan `n > 1`. Mengapa beberapa unit hidden membantu khusus untuk kelompok A yang berisi empat digit berbeda? (b) Bagaimana pengaruh `t` terhadap akurasi? Jelaskan apa yang hilang dari citra jika `t` terlalu kecil (`t = 0`) dan terlalu besar (`t = 12`). (c) Apakah `n` terbesar selalu terbaik? Jelaskan dengan data.

**Jawaban:**

Pada tabel pivot, n = 1 (setara satu ADALINE) hanya mencapai akurasi validasi 0.871 (t = 0) dan turun sampai 0.748 (t = 12), sedangkan n > 1 jauh lebih baik dan terbaik 0.921 pada t = 8, n = 8. (a) Beberapa unit hidden membantu karena kelompok A berisi empat digit berbeda (2, 3, 5, 8) yang membentuk beberapa daerah terpisah di ruang piksel sehingga satu garis lurus tidak cukup; dengan unit OR, setiap unit dapat mengkhususkan diri pada satu bentuk, terlihat pada Eksperimen D di mana unit 0 aktif pada digit 2 dan 3 (0.75 dan 0.74), unit 6 pada digit 5 (0.75), dan unit 1 pada digit 8 (0.50). (b) Threshold t berpengaruh besar: akurasi validasi tertinggi ada pada t = 0 sampai 8 (sekitar 0.91 sampai 0.92 untuk n ≥ 2), dan turun pada t = 12 (0.772, 0.856, dan 0.851 untuk n = 2, 4, 8). Jika t terlalu kecil (t = 0) semua piksel yang sedikit saja gelap dianggap tinta sehingga goresan menjadi tebal dan detail tipis bercampur dengan bayangan, sedangkan jika t terlalu besar (t = 12) hanya piksel paling gelap yang tersisa sehingga goresan menjadi putus dan informasi bentuk hilang. (c) n terbesar tidak selalu terbaik: pada t = 0, n = 2 dan n = 4 (0.911) sedikit lebih baik dari n = 8 (0.906), pada t = 4 n = 4 dan n = 8 sama (0.906), dan pada t = 12 n = 4 (0.856) lebih baik dari n = 8 (0.851); akurasi train n = 8 sudah 1.0 sementara validasi tidak naik, tanda mulai overfitting, dan Eksperimen D menunjukkan unit 2 tidak pernah aktif serta unit 7 hampir tidak pernah aktif sehingga sebagian unit tambahan tidak terpakai.

**Soal 3 - Unit output OR.**
(a) Tuliskan penurunan rumus `b_out` Anda di Tugas 3 beserta rentang nilai `b_out` yang masih valid. (b) Gunakan Eksperimen A: fungsi logika apa yang sebenarnya dihitung oleh unit output pada UTP sebelumnya (bias tetap 0,5) untuk `n = 4` dan `n = 8`? Apa akibatnya terhadap aturan MRI yang mengasumsikan output OR? (c) Gunakan Eksperimen B: apakah pemilihan kelas mana yang menjadi `+1` memengaruhi hasil? Jelaskan hubungannya dengan sifat asimetris OR (kelas positif = gabungan beberapa daerah, kelas negatif = irisan daerah negatif).

**Jawaban:**

(a) Pada Tugas 3, jika k dari n unit bernilai +1 maka jumlah masukan output adalah 0.5(2k - n) + b_out. Syarat OR adalah output -1 untuk k = 0, yaitu -0.5n + b_out < 0 sehingga b_out < 0.5n, dan output +1 untuk k = 1 (kasus terburuk), yaitu 0.5(2 - n) + b_out ≥ 0 sehingga b_out ≥ 0.5n - 1. Rentang b_out yang valid adalah 0.5n - 1 ≤ b_out < 0.5n, dan saya memilih titik tengah b_out = 0.5n - 0.5 yang berlaku untuk sembarang n ≥ 1 (uji otomatis mencetak True untuk n = 1, 2, 3, 4, 8). (b) Pada Eksperimen A dengan bias tetap 0.5, output menjadi +1 jika 0.5(2k - n) + 0.5 ≥ 0, yaitu k ≥ (n - 1)/2: untuk n = 4 hasilnya [-1, -1, 1, 1, 1] sehingga unit output baru bernilai +1 jika minimal 2 dari 4 unit aktif, dan untuk n = 8 hasilnya [-1, -1, -1, -1, 1, 1, 1, 1, 1] yaitu minimal 4 dari 8 unit aktif; jadi fungsi yang dihitung adalah fungsi ambang/mayoritas (bukan OR), bahkan untuk n = 1 output selalu +1. Akibatnya aturan MRI yang mengasumsikan OR menjadi tidak sesuai: saat t = +1 hanya satu unit yang diperbarui padahal output belum tentu berubah menjadi +1 (butuh beberapa unit aktif sekaligus), sehingga pelatihan sulit konvergen. (c) Eksperimen B menunjukkan pilihan kelas +1 memengaruhi hasil: misalnya pada n = 2 akurasi test 0.850 (A = +1) dibanding 0.912 (B = +1), pada n = 8 0.900 dibanding 0.926, dan pada n = 1 0.787 dibanding 0.771; selisihnya mengecil pada n yang lebih besar. Hal ini karena OR bersifat asimetris: kelas positif adalah gabungan beberapa daerah (tiap unit menutup satu daerah) sedangkan kelas negatif harus berupa irisan semua daerah negatif (daerah konveks tunggal), sehingga kelas yang bentuknya lebih beragam (kelompok A dengan empat digit berbeda) lebih sulit ditangani bila dijadikan kelas negatif atau bila unit tidak cukup.

**Soal 4 - Aturan MRI vs implementasi UTP sebelumnya.**
Gunakan Eksperimen C. (a) Identifikasi perbedaan aturan update `train_lama` dengan MRI yang benar (minimal tiga perbedaan, termasuk inisialisasi, unit yang diperbarui saat t = +1, besaran langkah, dan bias). (b) Jelaskan dampak masing-masing perbedaan terhadap hasil (akurasi dan rata-rata unit aktif). (c) Mengapa MRI memilih unit dengan $z\_in$ paling dekat ke 0 saat t = +1 (prinsip *minimal disturbance*)?

**Jawaban:**

(a) Perbedaan train_lama dengan MRI yang benar ada empat: inisialisasi bobot lama memakai rng.random (semua positif 0 sampai 1) dengan bias 0.5, sedangkan MRI memakai uniform [-0.1, 0.1] untuk bobot dan bias; saat t = +1 aturan lama memperbarui semua unit yang bernilai -1 dengan w += alpha * outer(x, 1 - z), sedangkan MRI hanya memperbarui satu unit J yang z_in-nya paling dekat ke 0; besaran langkah aturan lama tetap (alpha kali x, atau dikurangi alpha kali x saat t = -1) sedangkan MRI sebanding dengan selisih terhadap target, alpha * (1 - z_in) atau alpha * (-1 - z_in); dan bias pada aturan lama tidak pernah diubah (bahkan output memakai bias tetap 0.5 yang bukan OR) sedangkan MRI memperbarui bias setiap unit yang dikoreksi. (b) Dampaknya terlihat pada Eksperimen C: model lama hanya mencapai akurasi test 0.843 dan kesalahan per epoch tidak konvergen (222, 144, 150 pada epoch 1, 5, dan akhir), sedangkan MRI mencapai 0.914 dengan kesalahan 200, 78, 0 sehingga konvergen; rata-rata unit aktif per citra pada model lama 2.20 (banyak unit aktif bersamaan karena inisialisasi positif dan semua unit ikut diubah, sehingga unit menjadi mirip dan tidak berspesialisasi), sedangkan MRI hanya 0.67 (aktivasi jarang dan spesifik). Bias yang tidak diperbarui dan langkah tetap membuat batas keputusan kurang fleksibel dan mudah berosilasi. (c) MRI memilih unit dengan z_in paling dekat ke 0 saat t = +1 karena unit itu paling mudah diubah menjadi +1 dengan perubahan bobot terkecil (prinsip minimal disturbance), sehingga keputusan unit lain dan sampel yang sudah benar tidak terganggu, dan setiap unit terdorong menangani subkelompok digit yang berbeda.

**Soal 5 - Interpretasi model dan perbaikan.**
Gunakan Eksperimen D. (a) Interpretasikan pola bobot setiap unit hidden: digit mana yang "ditangkap" oleh masing-masing unit (rujuk tabel proporsi unit aktif per digit)? (b) Digit mana yang paling sering salah pada data bersih dan noisy, dan mengapa digit tersebut sulit (kaitkan dengan bentuk digit dari kelompok lawan)? (c) Usulkan **dua** perbaikan konkret yang realistis untuk meningkatkan akurasi atau ketahanan terhadap derau, beserta alasannya.

**Jawaban:**

(a) Berdasarkan tabel proporsi unit aktif per digit, unit 0 menangkap digit 2 dan 3 (aktif 0.75 dan 0.74), unit 4 terutama digit 2 (0.73), unit 3 menangkap sebagian digit 2 (0.37) dan 5 (0.25), unit 6 menangkap digit 5 (0.75), dan unit 1 menangkap digit 8 (0.50) dengan sebagian dukungan unit 5 (0.17); unit 2 tidak pernah aktif (0.00) dan unit 7 hampir tidak pernah aktif sehingga praktis hanya sekitar enam unit yang berfungsi, sedangkan pada digit kelompok B (1, 4, 7, 9) hampir semua unit tidak aktif (0.00 sampai 0.04) yang sesuai dengan perilaku OR. (b) Pada data bersih digit yang paling sering salah adalah 3 (0.222), lalu 8 (0.167), 1 (0.160), 5 (0.115), dan 9 (0.103), sedangkan pada data noisy yang tertinggi adalah 9 (0.276), 1 (0.260), 7 (0.204), 3 (0.167), dan 8 (0.150). Digit 3 sulit karena hanya satu unit utama (unit 0, 0.74) yang menangkapnya sehingga sebagian citra tidak diaktifkan oleh unit mana pun, dan lengkung kanan atasnya mirip goresan digit 7 dan 9 dari kelompok lawan; digit 8 mirip dengan 9 karena sama-sama memiliki lingkaran atas, dan digit 1 yang tipis dan bergeser mudah salah. Pada data noisy, digit 7 dan 4 yang tadinya tanpa kesalahan menjadi 0.204 dan 0.123 karena piksel garam (bernilai 16) menyalakan piksel yang cocok dengan pola unit positif. (c) Dua perbaikan realistis: pertama, tambahkan pra-pemrosesan median filter 3x3 sebelum binarisasi dan/atau latih dengan augmentasi data yang diberi derau salt-and-pepper, agar piksel rusak tidak langsung membalik fitur dan model terbiasa dengan derau; kedua, gunakan masukan skala keabuan yang dinormalisasi (piksel dibagi 16) alih-alih binarisasi keras, ditambah beberapa kali inisialisasi seed dan membuang unit mati (seperti unit 2 dan 7), sehingga informasi intensitas tidak hilang dan semua unit hidden dimanfaatkan untuk mengkhususkan diri pada digit yang sulit seperti 3 dan 8.